In [2]:
!pip install torch torchvision

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10

### Dataset and DataLoaders

In [3]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

In [4]:
trainset=CIFAR10(root='./data',train=True,download=True,transform=transform)
testset=CIFAR10(root='./data',train=False,download=True,transform=transform)

100%|██████████| 170M/170M [04:35<00:00, 618kB/s]


In [7]:
trainloader=DataLoader(trainset,batch_size=64,shuffle=True)
testloader=DataLoader(testset,batch_size=64,shuffle=False)

### Build CNN

In [14]:
from torch.optim import optimizer
class CNN(nn.Module):
  def __init__(self):
    super(CNN,self).__init__()

    self.conv_layers=nn.Sequential(
        nn.Conv2d(3,32,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(32,64,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(64,128,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2)

    )

    self.fc_layers=nn.Sequential(
        nn.Linear(4*4*128,512),
        nn.ReLU(),
        nn.Linear(512,10)
    )

  def forward(self,x):
    x=self.conv_layers(x)
    x=x.view(x.size(0),-1)
    x=self.fc_layers(x)
    return x

model=CNN()
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

### Training CNN

In [15]:
epochs=20
for epoch in range(epochs):
  epoch_training_loss=0.0
  for images,labels in trainloader:
    optimizer.zero_grad()
    outputs=model(images)
    loss=criterion(outputs,labels)
    loss.backward()
    optimizer.step()
    epoch_training_loss+=loss.item()
  print(f'Epoch {epoch} Training Loss: {epoch_training_loss/len(trainloader)}')

Epoch 0 Training Loss: 1.3289854655332882
Epoch 1 Training Loss: 0.8796321645264735
Epoch 2 Training Loss: 0.6943118010869112
Epoch 3 Training Loss: 0.5668837803289714
Epoch 4 Training Loss: 0.449240546892671
Epoch 5 Training Loss: 0.34695558859716596
Epoch 6 Training Loss: 0.24782736306948125
Epoch 7 Training Loss: 0.1827446860177895
Epoch 8 Training Loss: 0.1327440545363042
Epoch 9 Training Loss: 0.10925230002292739
Epoch 10 Training Loss: 0.09476076055715894
Epoch 11 Training Loss: 0.08020594075162564
Epoch 12 Training Loss: 0.07532245869887635
Epoch 13 Training Loss: 0.07221040145233226
Epoch 14 Training Loss: 0.06995473622844335
Epoch 15 Training Loss: 0.06564934766265418
Epoch 16 Training Loss: 0.057137131711820624
Epoch 17 Training Loss: 0.06191785417942573
Epoch 18 Training Loss: 0.06796303200845123
Epoch 19 Training Loss: 0.05560663061947717


In [18]:
# Save trained model weights

torch.save(model.state_dict(), "model.pth")

print("Model saved successfully!")

Model saved successfully!


### Evaluate Model

In [17]:
from functools import total_ordering
correct_labels=0
total_labels=0
with torch.no_grad():
  for images,labels in testloader:
    outputs=model(images)
    _, predicted=torch.max(outputs,1)
    total_labels+=labels.size(0)
    correct_labels+=(predicted==labels).sum().item()
print(f'Accuraccy = {correct_labels/total_labels*100}')

Accuraccy = 75.11
